# Dynamic Pricing — contextual algorithm comparison

Multi-algorithm comparison on the Dynamic Pricing game (contextual setting) for several numbers of
prices (arms), in the style of `run_comparison_at.py` / `run_comparison_at_v2.py`.

Dynamic pricing (Kleinberg & Leighton, 2003) is the canonical "hard" example of a partial-monitoring game in
Cesa-Bianchi, Lugosi & Stoltz (2006) and Bartók, Pál & Szepesvári (2011), and the benchmark game of the CBP
(Bartók, Zolghadr & Szepesvári, 2012), PM-DMED (Komiyama et al., 2015) and TSPM (Tsuchiya et al., 2020)
experiments; the CBP paper runs it with $N = M = 5$ and no-sale cost $c = 2$.

## Model

**Game** (`games.dynamic_pricing(n_prices, c)`).  A seller posts one of $N$ prices, the buyer has a hidden
valuation among $M = N$ levels, and a sale happens iff the price does not exceed the valuation.

* **Action** $a \in \{0, \dots, N-1\}$: post the price $p_a = a + 1$.
* **Outcome** $y \in \{0, \dots, M-1\}$: the buyer's valuation is $v_y = M - y$ (outcome $0$ is the highest
  valuation, so that $N = M = 2$ has the structure of Apple Tasting).
* **Loss** (matrix $L$): lost revenue if the item sells, the constant $c$ if it does not,
$$
\ell(a, y) \;=\; L_{a,y} \;=\;
\begin{cases}
v_y - p_a & \text{if } p_a \le v_y \quad (\text{sold}),\\[2pt]
c         & \text{if } p_a > v_y   \quad (\text{not sold}),
\end{cases}
\qquad\text{reward } r(a, y) = -\ell(a, y).
$$
* **Feedback** (matrix $H$): the seller only observes whether the item sold,
$$
h(a, y) \;=\; H_{a,y} \;=\; \mathbb{1}\{p_a \le v_y\} \in \{0, 1\}.
$$
  Equivalently, action $a$ has the signal matrix $S_a \in \{0,1\}^{\sigma_a \times M}$ with
  $(S_a)_{s, y} = \mathbb{1}\{H_{a,y} = s\}$; the outcome distribution $q \in \Delta_M$ induces the signal
  distribution $S_a q$.  For $N = 2$ the high price reveals the valuation ($S_1$ is invertible; with $c = 1$ the
  game is exactly Apple Tasting), for $N \ge 3$ no single price does (globally but not locally observable game).

**Context and valuation model** (`synthetic_data.LinearContexts` with `n_outcomes` valuation levels).  At each round the customer's raw
features are $u_t \sim \mathrm{Unif}([0,1]^d)$ with $d = 10$; the algorithms receive the standardised context
$x_t = (u_t - \mu) / \sigma \in \mathbb{R}^d$ (coordinate-wise mean and standard deviation of $u$).
The valuation score is the linear function
$$
s_t \;=\; w^\top u_t \in [0, 1], \qquad w = (0.6,\, 0.4,\, 0,\, \dots,\, 0)
$$
(the weights of `run_comparison_at.py`, padded with uninformative features so that $d \ge 5$ as required by
CBPside / RandCBPside), and the conditional outcome distribution is binomial in the score, counted from the
highest level,
$$
q_t(j) \;=\; \Pr(y_t = j \mid x_t) \;=\; \binom{M-1}{j}\, s_t^{\,M-1-j}\,(1 - s_t)^{\,j},
\qquad j = 0, \dots, M-1,
$$
so that the expected valuation is affine in $s_t$ and, for $M = 2$, $q_t = (s_t,\, 1 - s_t)$, the original
two-outcome `LinearContexts` used by the Apple Tasting and Label Efficient experiments.

**Protocol** (`evaluation_contextual.Evaluation_contextual`, as in the runner scripts).  The outcome is the
most likely valuation level, $y_t = \arg\max_j q_t(j)$, i.e. the equal-width bin of the score $s_t$
(for $M = 2$ the code base's original rule $y_t = \mathbb{1}\{s_t \ge 1/2\}$ is kept), the learner plays
$a_t$, observes $h(a_t, y_t)$, and the reported quantity is the cumulative regret
$$
R_T \;=\; \sum_{t=1}^{T} \Big[\, \ell(a_t, y_t) \;-\; \min_{a}\, \ell(a, y_t) \Big].
$$

**Algorithms** (all from the code base, hyper-parameters of `run_comparison_at_v2.py`):
  - Random           — uniform baseline
  - PGTS             — Polya-Gamma Thompson Sampling            ($N = 2$ only: two-action implementation)
  - PGTS-multi       — multi-arm PG-TS (`PGTS_multiarm.PGTSMultiArm`): one Pólya-Gamma logistic model of the
                       sold / not-sold signal per price (Dumitrascu et al., 2018, Algorithm 2), Thompson draw,
                       $\hat q_t$ by least squares over the prices' signals, price $= \arg\min_a (L \hat q_t)_a$
  - STAP-Helmbolt    — Helmbold et al.                          ($N = 2$ only: two-action implementation)
  - CBPside          — Confidence Bounds for PM (deterministic)
  - RandCBPside      — Confidence Bounds for PM (randomized)
  - SquareCB.PMSide  — IGW + water-transfer (this work).  The outcome estimate is
    $\hat q_t = S_k^{-1} \hat y_{k}(x_t)$ from the informative price $k$ when one exists ($N = 2$) and otherwise
    the least-squares solution $\hat q_t = \arg\min_q \sum_a \lVert S_a q - \hat y_a(x_t) \rVert^2$ over all prices,
    where $\hat y_a(x_t)$ is the ridge-regression prediction of the signal of price $a$.

Output (in `RESULTS_DIR`):
  - `comparison_dp_results.npy`   (dict of {n_prices: {name: (N_SEEDS, HORIZON) array}})
  - `comparison_dp_regret.png`    (one panel per number of prices)


In [ ]:
import os
import io
import contextlib
import numpy as np
import matplotlib.pyplot as plt

from games import dynamic_pricing
from synthetic_data import LinearContexts
from evaluation_contextual import Evaluation_contextual
from squarecb_pmside import SquareCBPMSide
from STAP_Helmbolt import STAP_Helmbolt
from PGTS import PGTS
from PGTS_multiarm import PGTSMultiArm
from random_algo import Random
from cbpside import CBPside
from randcbpside import RandCPBside

In [ ]:
# ── Experiment settings ────────────────────────────────────────────────────────
HORIZON       = 3000
N_SEEDS       = 5
N_PRICES_LIST = [2, 3, 5]   # number of prices (arms); the number of valuation levels is the same
C             = 2.0         # no-sale cost c (c = 2 in the CBP experiments; N = 2, c = 1 is Apple Tasting)
D             = 10          # context dimension — must be >= 5 for CBPside/RandCBPside
GAMMA         = 1.0         # SquareCB.PMSide: effective rate gamma_t = GAMMA * sqrt(t)
LBD           = 0.05        # ridge regularisation (all ridge-regression algorithms)
ALPHA         = 1.01        # CBPside / RandCBPside confidence-bound exponent

# Outcome-probability weights of run_comparison_at.py, padded with uninformative features
W = np.zeros(D)
W[0], W[1] = 0.6, 0.4

# RandCBPside-specific
SIGMA   = 5                 # bandwidth for the randomised confidence level
K       = 10                # number of quantisation levels
EPSILON = 10e-7             # exploration probability floor

RESULTS_DIR = "results"     # where the raw results and the plot are saved

In [ ]:
# ── Algorithm registry ────────────────────────────────────────────────────────
# Each entry: (display_name, factory_fn)
# factory_fn() returns a fresh algorithm instance.
# PGTS and STAP_Helmbolt are written for the two-action game (action 1 informative),
# so they are only included for N = 2; PGTSMultiArm is the multi-arm PG-TS.
def make_algos(game, n_prices):
    algos = [
        ("Random",
         lambda: Random(game, HORIZON)),
    ]
    if n_prices == 2:
        algos += [
            ("PGTS",
             lambda: PGTS(game, D)),

            ("STAP-Helmbolt",
             lambda: STAP_Helmbolt(game, D)),
        ]
    algos += [
        ("PGTS-multi",
         lambda: PGTSMultiArm(game, D)),      # PG-TS with one logistic model per price (any N)

        ("CBPside",
         lambda: CBPside(game, D, ALPHA, LBD)),

        ("RandCBPside",
         lambda: RandCPBside(game, D, ALPHA, LBD, SIGMA, K, EPSILON)),

        ("SquareCB.PMSide",
         lambda: SquareCBPMSide(game, d=D, gamma=GAMMA, lbd=LBD,
                                use_water_transfer=(n_prices > 2))),   # identity is enough for 2 actions
    ]
    return algos

In [ ]:
# ── Run all algorithms for every number of prices ─────────────────────────────
all_results = {}

for n_prices in N_PRICES_LIST:
    # Game / evaluator (the geometry LPs of the game constructor print Gurobi messages)
    with contextlib.redirect_stdout(io.StringIO()):
        game        = dynamic_pricing(n_prices, C)
        context_gen = LinearContexts(W, n_outcomes=n_prices)
    evaluator = Evaluation_contextual(HORIZON)
    print(f"\n=== Dynamic Pricing: N = {n_prices} prices, c = {C:g}  |  neighbouring pairs {game.mathcal_N} ===")

    results = {}
    for name, factory in make_algos(game, n_prices):
        runs = []
        print(f"\nRunning {name}", end="", flush=True)

        for seed in range(N_SEEDS):
            # Suppress the verbose per-step prints from evaluation_contextual and the
            # algorithms (and the Gurobi messages of CBPside's constructor) so the
            # output stays readable.
            with contextlib.redirect_stdout(io.StringIO()):
                alg = factory()
                # reset() initialises contexts for algorithms that don't do it in __init__
                # (e.g. STAP_Helmbolt, RandCBPside).  Safe to call on all algorithms.
                alg.reset()
                cr = evaluator.eval_policy_once(alg, game, job=(context_gen, seed))

            runs.append(cr)
            print(f" {seed}", end="", flush=True)

        runs = np.array(runs)   # shape (N_SEEDS, HORIZON)
        results[name] = runs
        mean_final = runs.mean(axis=0)[-1]
        print(f"  |  mean cumulative regret @ T={HORIZON}: {mean_final:.2f}")

    all_results[n_prices] = results

In [ ]:
# ── Save raw results ───────────────────────────────────────────────────────────
os.makedirs(RESULTS_DIR, exist_ok=True)
np.save(os.path.join(RESULTS_DIR, "comparison_dp_results.npy"), all_results)
print(f"Raw results saved to {RESULTS_DIR}/comparison_dp_results.npy")

In [ ]:
# ── Plot (one panel per number of prices) ──────────────────────────────────────
t      = np.arange(1, HORIZON + 1)
COLORS = {
    "Random":          "#aaaaaa",
    "PGTS":            "#e07b54",
    "PGTS-multi":      "#8c564b",
    "STAP-Helmbolt":   "#5b8db8",
    "CBPside":         "#c07ad4",
    "RandCBPside":     "#d4a017",
    "SquareCB.PMSide": "#3a9668",
}
STYLES = {
    "Random":          "--",
    "PGTS":            "-.",
    "PGTS-multi":      (0, (6, 2, 1, 2)),    # long dash-dot
    "STAP-Helmbolt":   ":",
    "CBPside":         (0, (3, 1, 1, 1)),   # dash-dot-dot
    "RandCBPside":     (0, (5, 2)),          # loose dash
    "SquareCB.PMSide": "-",
}

fig, axes = plt.subplots(1, len(N_PRICES_LIST), figsize=(5.5 * len(N_PRICES_LIST), 4.5), squeeze=False)

for ax, n_prices in zip(axes[0], N_PRICES_LIST):
    for name, runs in all_results[n_prices].items():
        mean  = runs.mean(axis=0)
        std   = runs.std(axis=0)
        color = COLORS[name]
        ls    = STYLES[name]
        ax.plot(t, mean, label=name, color=color, linewidth=1.8, linestyle=ls)
        ax.fill_between(t, mean - std, mean + std, alpha=0.15, color=color)

    ax.set_xlabel("Round $t$", fontsize=12)
    ax.set_ylabel("Cumulative regret", fontsize=12)
    ax.set_title(f"N = {n_prices} prices", fontsize=13)
    ax.legend(fontsize=9)
    ax.grid(True, linewidth=0.35, alpha=0.6)

fig.suptitle("Dynamic Pricing — contextual algorithm comparison", fontsize=13)
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "comparison_dp_regret.png"), dpi=150)
print(f"Plot saved to {RESULTS_DIR}/comparison_dp_regret.png")